In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path
import os

### Make manifest for alternate architectures

In [2]:
archs_to_skip = [0, 3, 5] # arch configs that didn't train (too large, didn't converge, etc.)
path_to_arch_configs = [config for config in Path("config/arch_search").glob('*.yaml') 
                        if not any([f'_{i}' in config.stem for i in archs_to_skip])]

print("N archs to test:", len(path_to_arch_configs))
print("Configs (as posix paths)")
print("\n".join( str(path) for path in path_to_arch_configs))

test_condition_ixs = np.arange(0,61) # 60 tests + clean 

condition_mapping = itertools.product(path_to_arch_configs, test_condition_ixs)


N archs to test: 7
Configs (as posix paths)
config/arch_search/word_task_v09_4MGB_ln_first_arch_1.yaml
config/arch_search/word_task_v09_4MGB_ln_first_arch_2.yaml
config/arch_search/word_task_v09_4MGB_ln_first_arch_4.yaml
config/arch_search/word_task_v09_4MGB_ln_first_arch_6.yaml
config/arch_search/word_task_v09_4MGB_ln_first_arch_7.yaml
config/arch_search/word_task_v09_4MGB_ln_first_arch_8.yaml
config/arch_search/word_task_v09_4MGB_ln_first_arch_9.yaml


In [4]:
# get latest checkpoint for each config 
ckpt_dir = Path('attn_cue_models')

ckpt_dict = {}
for config in path_to_arch_configs:
    ckpt_files = list(ckpt_dir.glob(f'{config.stem}/checkpoints/*.ckpt'))
    if len(ckpt_files) == 0:
        print(f"Warning: no ckpt found for {config.stem}")
        continue
    latest_ckpt = max(ckpt_files, key=lambda x: x.stat().st_mtime)
    ckpt_dict[config.stem] = latest_ckpt

In [5]:
## Hand pick best val ckpts where different from latest - epoch number reflects run, not true global step number 

ckpt_dict["word_task_v09_4MGB_ln_first_arch_1"] = '/om2/user/imgriff/projects/torch_2_aud_attn/attn_cue_models/word_task_v09_4MGB_ln_first_arch_1/checkpoints/epoch=2-step=40080.ckpt'
ckpt_dict["word_task_v09_4MGB_ln_first_arch_2"] = '/om2/user/imgriff/projects/torch_2_aud_attn/attn_cue_models/word_task_v09_4MGB_ln_first_arch_2/checkpoints/epoch=2-step=32806-v4.ckpt'
ckpt_dict["word_task_v09_4MGB_ln_first_arch_6"] = '/om2/user/imgriff/projects/torch_2_aud_attn/attn_cue_models/word_task_v09_4MGB_ln_first_arch_6/checkpoints/epoch=2-step=32806-v3.ckpt'
ckpt_dict["word_task_v09_4MGB_ln_first_arch_8"] = '/om2/user/imgriff/projects/torch_2_aud_attn/attn_cue_models/word_task_v09_4MGB_ln_first_arch_8/checkpoints/epoch=0-step=4000.ckpt'


In [18]:
ckpt_dict
parent_dir = Path('attn_cue_models')
for k,v in ckpt_dict.items():
    v = Path(v)
    full_path = parent_dir / f"{k}/{v.parent.stem}/{v.name}"
    if not full_path.exists():
        print(f"Warning: {full_path} does not exist") 
    print(f"{k}/{v.parent.stem}/{v.name}")

word_task_v09_4MGB_ln_first_arch_1/checkpoints/epoch=2-step=40080.ckpt
word_task_v09_4MGB_ln_first_arch_2/checkpoints/epoch=2-step=32806-v4.ckpt
word_task_v09_4MGB_ln_first_arch_4/checkpoints/epoch=0-step=8000-v4.ckpt
word_task_v09_4MGB_ln_first_arch_6/checkpoints/epoch=2-step=32806-v3.ckpt
word_task_v09_4MGB_ln_first_arch_7/checkpoints/epoch=0-step=8000-v7.ckpt
word_task_v09_4MGB_ln_first_arch_8/checkpoints/epoch=0-step=4000.ckpt
word_task_v09_4MGB_ln_first_arch_9/checkpoints/epoch=0-step=4000-v5.ckpt


In [6]:
test_condition_manifest = {}
for ix, (config, test_idx) in enumerate(condition_mapping):
    test_condition_manifest[ix] = {
        'config_path': config,
        'ckpt_path': ckpt_dict[config.stem],
        'test_idx': test_idx
    }



In [7]:
outdir = Path("swc_test_manifests")
outdir.mkdir(exist_ok=True, parents=True)

with open(outdir / "arch_search_configs_11_17_2024_all_conds_w_best_ckpts.pkl", 'wb') as f:
    pickle.dump(test_condition_manifest, f, protocol=pickle.HIGHEST_PROTOCOL)
#     match_human_pilot_conds = pickle.load(f)

In [50]:
isinstance(test_condition_manifest[1], dict)

True

In [49]:
### Print n tests for job array ix size 

n_tests = len(test_condition_manifest)
print(f"N tests: {n_tests}")
print(f"Array_ids: 0-{n_tests-1}")


N tests: 427
Array_ids: 0-426
